# 07 — Activity Diversity and Category Mix

These reports describe recorded task starts, not health or productivity. Changes may reflect seasons, travel, or recording habits. Filters use exact names; `None` includes all values. Choose a task type to focus rolling and seasonal comparisons on one activity. Missing records are not proof of inactivity. Engagement-domain reports (including brief section 5) are deferred with the section 3 schema enhancement.


In [ ]:
%run pathutils.ipynb
%run database.ipynb
%run export.ipynb
%run reporting-utils.ipynb

import sys
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(Path(get_project_root_folder()) / "reports"))
from support.activity_analysis import (
    ReportPeriod,
    filter_history,
    monthly_profile,
)

In [ ]:
# Inclusive dates; END_DATE is also the as-of date. Use finite ISO dates.
START_DATE = "2024-01-01"
END_DATE = date.today().isoformat()
CATEGORY = None
TASK_TYPE = None
LOCATION = None
ROLLING_WINDOW = 90
DORMANCY_THRESHOLD = 90

In [ ]:
period = ReportPeriod(START_DATE, END_DATE, CATEGORY, TASK_TYPE, LOCATION)
# Load earlier observations for rolling windows, continuity and the full backlog.
query = construct_query("task-history.sql", {"START-DATE": "0001-01-01", "END-DATE": END_DATE})
history = filter_history(query_data(query), period)
filters = f"category={CATEGORY or 'All'}, type={TASK_TYPE or 'All'}, location={LOCATION or 'All'}"
period_label = f"{START_DATE} to {END_DATE}\n{filters}"
print(period_label)

## Monthly breadth and category proportions

Empty months have zero counts and undefined category shares. Partial boundary months use only the selected dates; nonempty months total 100%.


In [ ]:
diversity, category_mix = monthly_profile(history, period)
display(diversity, category_mix)

In [ ]:
if not diversity.empty and len(diversity.columns) > 1:
    figure, axes = plt.subplots(figsize=CHART_FIGURE_SIZE)
    diversity.plot(
        x="Month",
        y=["Distinct Task Types", "Active Categories"],
        kind="line",
        stacked=False,
        ax=axes,
    )
    format_chart(
        figure, axes, f"Monthly activity breadth — {period_label}", "Month", "Count", grid_axis="y"
    )
    export_chart(get_export_folder_path(), "activity-diversity", "png", figure)
    plt.show()
else:
    print("No observations available for this chart.")

In [ ]:
if not category_mix.empty and len(category_mix.columns) > 1:
    figure, axes = plt.subplots(figsize=CHART_FIGURE_SIZE)
    category_mix.assign(Month=category_mix["Month"].dt.strftime("%Y-%m")).plot(
        x="Month", y=None, kind="bar", stacked=True, ax=axes
    )
    format_chart(
        figure,
        axes,
        f"Monthly category mix — {period_label}",
        "Month",
        "Percentage of tasks",
        grid_axis="y",
    )
    axes.set_ylim(0, 100)
    export_chart(get_export_folder_path(), "category-mix", "png", figure)
    plt.show()
else:
    print("No observations available for this chart.")

## Export

Tables are exported as XLSX and charts as PNG to `data/reports/` (or the runner’s configured destination).


In [ ]:
export_to_spreadsheet(
    get_export_folder_path(),
    "activity-diversity-and-mix.xlsx",
    {"Diversity": diversity, "Category Mix": category_mix},
)